# TN0 — đối chiếu pipeline đồ án với pipeline MobiVital

Chạy một mạch trên Colab. Không cần notebook nào chạy trước.

## Ba kiểm tra

| | câu hỏi | bắt buộc đạt |
|---|---|---|
| **TN0a** | Tệp lựa chọn kênh tác giả cung cấp có tái hiện điểm công bố (~0.819) không? | có |
| **TN0b** | Cùng tệp trọng số `.pth`, pipeline đồ án có chọn đúng 537/537 kênh giống pipeline MobiVital không? | có |
| **TN0c** | Train lại LSTM từ đầu thì đạt mức nào? | không — chỉ tham khảo |

## Hai pipeline

| | code | các tệp chính |
|---|---|---|
| **pipeline MobiVital** | tác giả cung cấp | `inference/evaluate.py`, `inference/mobivital_gen.py`, `training/autoreg_training.py` |
| **pipeline đồ án** | trong `src/`, gọi qua `scripts/run_tn0.py` | `scoring.py`, `training.py`, `results.py` |

Pipeline đồ án tồn tại vì code MobiVital chỉ chạy LSTM — `inference/mobivital_gen.py` dòng 152 ghi cứng `LSTMMultiStep(...)`, không hỗ trợ TCN.

## Notebook này chỉ gọi lệnh

```
notebook  ->  trinh bay va goi lenh
scripts/  ->  dieu khien tung thuc nghiem
src/      ->  model, train, chon kenh, tinh diem
```


## 1. Chuẩn bị Colab, Drive và mã nguồn


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR || git -C /content/UWB_RADAR pull -q origin main
%cd /content/UWB_RADAR
!git clone -q https://github.com/nesl/mobivital-public.git external/mobivital || true
!pip install -q einops


In [ ]:
!echo "commit đồ án     : $(git rev-parse --short HEAD)"
!echo "commit MobiVital : $(git -C external/mobivital rev-parse --short HEAD)"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader


## 2. Chuẩn bị và kiểm tra dữ liệu chung

Một bản CSV duy nhất, đặt trong thư mục MobiVital. Hai pipeline đọc chung bản đó:

```
external/mobivital/dataset/mobivital/tripod/*.csv     1874 tep
        |
        +-- prep_breath_final.py cua tac gia  ->  external/mobivital/data_final/*.npy
        |
        +-- scripts/make_npz.py cua do an     ->  data/processed/by_user/*.npz
                        |
              check_data.py: phai khop TUNG BYTE
                        |
              make_windows.py: cat cua so train
```

`data/` chỉ chứa thứ pipeline đồ án sinh ra. Mỗi script tự kiểm tra đủ tệp và bỏ qua nếu đã xong, nên chạy lại được sau khi Colab ngắt phiên.


In [ ]:
!python scripts/download_dataset.py


In [ ]:
!python scripts/mobivital/setup_dataset.py


In [ ]:
%cd /content/UWB_RADAR/external/mobivital
!python dataset_preparation/prep_breath_final.py
!ls -la data_final/


In [ ]:
%cd /content/UWB_RADAR
!python scripts/make_npz.py


In [ ]:
!python scripts/check_data.py


In [ ]:
!python scripts/make_windows.py


## 3. Chạy pipeline MobiVital

Đúng lệnh trong README của tác giả, chạy từ trong thư mục repo của họ, không sửa dòng code nào:

```
autoreg_training.py  ->  mobivital_gen.py  ->  evaluate.py
   train                  chon kenh             tinh diem
```

`mobivital_gen.py` luôn ghi tệp lựa chọn kênh ra đúng một tên `inference/methods/tripod_mobivital_pre_invert_0.9.txt`, mà tên đó **trùng một tệp có sẵn trong repo tác giả**. Nên sau mỗi lần chạy gọi `scripts/mobivital/save_txt.py` để chép kết quả ra `results/` rồi khôi phục tệp gốc.


### TN0a — tính điểm từ tệp lựa chọn kênh tác giả cung cấp

Chưa đụng model: kênh và phép biến đổi đã ghi sẵn trong tệp. Cờ `-d` trỏ sang thư mục có 52 lối tắt tên cũ.


In [ ]:
%cd /content/UWB_RADAR/external/mobivital
!cp inference/methods/tripod_mobivital_pre_invert_0.9.txt ../../results/TN0a.txt
!python -m inference.evaluate \
    -m tripod_mobivital_pre_invert_0.9.txt \
    -d ./dataset/mobivital/tripod_old_names \
    --save_file scores_TN0a.csv
!cp inference/methods/scores_TN0a.csv ../../results/


### TN0b — chọn kênh bằng tệp trọng số tác giả phát hành

Mỗi buổi ghi: dựng 240 ứng viên (120 kênh khoảng cách × 2 phép biến đổi) → lọc `invert_detector` → cắt 52 cửa sổ mỗi ứng viên → LSTM dự báo → chọn ứng viên có tổng Pearson cao nhất. Bước chọn **không nhìn nhịp thở thật**.


In [ ]:
!python -m inference.mobivital_gen
!python ../../scripts/mobivital/save_txt.py TN0b
!python -m inference.evaluate -m TN0b.txt --save_file scores_TN0b.csv
!cp inference/methods/scores_TN0b.csv ../../results/


### TN0c — train lại LSTM từ đầu

Cấu hình lấy từ `checkpoints/optimal_params.json` của tác giả: 20 epoch, Adam lr 1e-4, batch 64, MSE. Cờ `--model_name` để tệp trọng số mới không đè lên tệp tác giả phát hành.


In [ ]:
!python -m training.autoreg_training --model_name lstm_retrained


In [ ]:
!python -m inference.mobivital_gen --model_name lstm_retrained
!python ../../scripts/mobivital/save_txt.py TN0c
!python -m inference.evaluate -m TN0c.txt --save_file scores_TN0c.csv
!cp inference/methods/scores_TN0c.csv ../../results/


In [ ]:
%cd /content/UWB_RADAR
!echo "Tệp bị sửa trong repo MobiVital (phải trống):"
!git -C external/mobivital status --porcelain


## 4. Chạy pipeline đồ án

Ba kiểm tra như trên, bằng code trong `src/`, gọi qua `scripts/run_tn0.py`:

| kiểm tra | pipeline MobiVital | pipeline đồ án |
|---|---|---|
| tính điểm từ tệp lựa chọn kênh | `inference/evaluate.py` | `scoring.score_from_txt` |
| chọn kênh từ tệp trọng số | `inference/mobivital_gen.py` | `scoring.score_all` |
| train lại LSTM | `training/autoreg_training.py` | `training.train` |

Mỗi `--case` thêm đúng một bộ phận, nên bộ phận nào sai thì lộ ra ở đúng kiểm tra đó.


In [ ]:
!python scripts/run_tn0.py --case a


In [ ]:
!python scripts/run_tn0.py --case b


In [ ]:
!python scripts/run_tn0.py --case c


## 5. So sánh hai pipeline

- **TN0a** phải khớp: cùng tệp lựa chọn kênh, cùng cách tính điểm.
- **TN0b** phải khớp 537/537 kênh và chênh lệch điểm dưới `1e-9`: cùng tệp trọng số, không có gì ngẫu nhiên.
- **TN0c** chỉ tham khảo: hai vòng train khác nhau ở thứ tự xáo trộn dữ liệu.

Lệnh dưới trả mã lỗi khác 0 nếu có kiểm tra bắt buộc không đạt.


In [ ]:
!python scripts/run_tn0.py --compare


## 6. Lưu toàn bộ kết quả


In [ ]:
!mkdir -p /content/drive/MyDrive/mobivital
!tar -czf /content/drive/MyDrive/mobivital/tn0.tar.gz results runs/tn0
!ls -la /content/drive/MyDrive/mobivital/tn0.tar.gz
!ls results
